<small>Reconocimiento de Patrones · Licenciatura en Ciencias de la Computación · Universidad de Sonora<br>
Julio Waissman · <a href="https://ml-unison.github.io">ml-unison.github.io</a> · Licencia CC BY-SA 4.0</small>

---

# Laboratorio 8 · Un servicio de predicción vigilado

**Unidad 8**

Entrenar un modelo es la mitad del trabajo. La otra mitad empieza el día en que el modelo se pone **en producción**: alguien le manda datos nuevos, espera una respuesta en milisegundos y confía en que esa respuesta sea tan buena como la que vimos en la validación. Pero el mundo no se queda quieto: cambian los clientes, los precios, el clima, la manera en que se usa un servicio. Un modelo que nadie vigila se degrada en silencio.

En este laboratorio vas a construir, desde cero, las piezas de un servicio de predicción **vigilado**:

1. el **índice de estabilidad poblacional** (PSI), que mide cuánto se movió una distribución;
2. el estadístico de **Kolmogorov–Smirnov** de dos muestras, vectorizado;
3. un **reporte de cambio** por atributo, que combina los dos;
4. una **API con FastAPI** que valida la entrada con Pydantic y devuelve la predicción junto con la versión del modelo;
5. una política de **reentrenamiento con ventana móvil**, y su efecto en el error.

**Cómo se trabaja.** Completa las celdas marcadas con `...`. Después de cada pregunta hay una celda `grader.check("p1")` que verifica tu respuesta: si algo falla, te dice qué. Las preguntas marcadas como *reflexión* se responden por escrito en la celda indicada.

**Reglas.** Dentro de tus funciones de las preguntas 1 a 4 usa solo numpy y pandas: nada de `scipy.stats`, que aquí se usa únicamente para comparar. Las operaciones deben ser vectorizadas: no recorras los datos con ciclos (el ciclo sobre los atributos en la pregunta 4 y sobre los meses en la pregunta 6 sí está permitido).

**Cómo se entrega.** Ejecuta la libreta de principio a fin (*Entorno de ejecución → Reiniciar y ejecutar todo*), revisa que no haya errores y descarga el archivo `.ipynb`.

Las verificaciones de esta libreta son solo una parte de la revisión: al calificar se usan pruebas adicionales.

In [ ]:
import os, sys

RAMA = "rediseno-2027"  # rama del repositorio del curso de la que se toman paquete, datos y pruebas
os.environ["RDP_RAMA"] = RAMA
if "google.colab" in sys.modules:
    %pip install -q otter-grader "rdp @ git+https://github.com/ml-unison/ml-unison.github.io@{RAMA}#subdirectory=paquete"

In [ ]:
# Verificador de respuestas (otter-grader). Usa las pruebas públicas del laboratorio:
# la copia local si existe; si no (en Colab), las descarga del repositorio del curso.
import pathlib, urllib.request
import otter

PRUEBAS = pathlib.Path("tests") / "u08-lab08-servicio-monitoreo"
if not PRUEBAS.exists():
    PRUEBAS.mkdir(parents=True)
    for _p in ['p1.py', 'p2.py', 'p3.py', 'p4.py', 'p5.py', 'p6.py']:
        urllib.request.urlretrieve(
            f"https://raw.githubusercontent.com/ml-unison/ml-unison.github.io/{RAMA}/unidades/08-mlops/laboratorios/tests/u08-lab08-servicio-monitoreo/{_p}",
            PRUEBAS / _p)
grader = otter.Notebook(tests_dir=str(PRUEBAS))

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q fastapi httpx
    pass

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

import rdp

rdp.estilo.usar()
warnings.filterwarnings("ignore", message=".*httpx.*")  # aviso interno de starlette al importar TestClient

## Los datos y el problema

Volvemos a las **bicicletas de Washington** de la unidad 4: 17,379 horas del sistema de renta Capital Bikeshare de Washington, D. C., de enero de 2011 a diciembre de 2012 (H. Fanaee-T y J. Gama, *Progress in Artificial Intelligence*, 2014; repositorio UCI, licencia CC BY 4.0). Cada renglón es una hora; `total` es el número de bicicletas rentadas en esa hora. Usaremos seis atributos que se conocen **antes** de que la hora transcurra (el pronóstico del clima, el calendario):

- `hora` (0 a 23) y `laboral` (1 si no es fin de semana ni feriado);
- `clima`: 1 despejado, 2 nublado o neblina, 3 lluvia o nieve ligera, 4 lluvia o nieve intensa;
- `temp` en °C, `humedad` en % y `viento` en km/h.

**El problema.** Imagina que eres parte del equipo de datos del sistema de bicicletas y que, al terminar 2011, entrenaste un modelo que predice las rentas de cada hora; con él se decide cuántas bicicletas redistribuir entre estaciones. El modelo se pone en producción el 1 de enero de 2012 y nadie lo vuelve a tocar. **¿Sigue funcionando un año después? ¿Cómo nos habríamos enterado a tiempo?** Vamos a simular 2012 mes a mes, como si los datos llegaran en vivo.

El modelo es un *boosting* de árboles con pérdida de Poisson (las rentas son conteos), entrenado con todo 2011.

In [ ]:
bicis = rdp.datos.cargar("bicis_washington")
bicis["fecha"] = pd.to_datetime(bicis["fecha"])
bicis["periodo"] = bicis["fecha"].dt.to_period("M")
VARIABLES = ["hora", "laboral", "clima", "temp", "humedad", "viento"]


def fabrica_modelo():
    """Un modelo nuevo, sin entrenar, siempre con la misma configuración."""
    return HistGradientBoostingRegressor(loss="poisson", max_iter=100, early_stopping=False, random_state=0)


b2011 = bicis[bicis["fecha"].dt.year == 2011]
b2012 = bicis[bicis["fecha"].dt.year == 2012]
modelo = fabrica_modelo().fit(b2011[VARIABLES], b2011["total"])

print(f"2011: {len(b2011):,} horas · 2012: {len(b2012):,} horas")
print(f"error absoluto medio en 2011 (entrenamiento): {mean_absolute_error(b2011['total'], modelo.predict(b2011[VARIABLES])):.1f} rentas por hora")
b2011.head()

# Parte A · ¿Se movió la distribución?

## 1. Cubetas definidas por los cuantiles de la referencia

Para comparar la distribución de un atributo en dos momentos, la **referencia** (lo que vio el modelo al entrenarse) y la **actual** (lo que llega en producción), el índice de estabilidad poblacional divide la recta en $k$ cubetas y compara qué fracción de cada muestra cae en cada una.

Las cubetas se definen con los **cuantiles de la referencia**: los cortes son $c_j = $ cuantil $j/k$ de la referencia, para $j = 1, \dots, k-1$. Así, en la referencia cada cubeta tiene cerca de $1/k$ de los datos, y las cubetas son angostas donde hay muchos datos y anchas donde hay pocos. Con atributos discretos (la hora, el clima) varios cortes coinciden: nos quedamos con los cortes **distintos** (`np.unique`), y entonces hay menos de $k$ cubetas.

Con los cortes ordenados $c_1 < \dots < c_m$, un valor $x$ cae en la cubeta $i$ si exactamente $i$ cortes son $\le x$; es decir, cubeta 0 si $x < c_1$, cubeta $i$ si $c_i \le x < c_{i+1}$ y cubeta $m$ si $x \ge c_m$. En total hay $m + 1$ cubetas, y esa regla es justamente lo que calcula `np.searchsorted(cortes, x, side="right")`. Para contar cuántos valores caen en cada cubeta sirve `np.bincount(..., minlength=m + 1)`.

**Pregunta 1.** Escribe `proporciones(referencia, actual, k=10)`, que devuelve la tupla `(q, p)`: `q` es el arreglo con la fracción de la **referencia** que cae en cada cubeta y `p` la fracción de la muestra **actual**, con las cubetas definidas por los cortes distintos `np.unique(np.quantile(referencia, [1/k, 2/k, ..., (k-1)/k]))` y la regla de arriba. Las dos fracciones suman 1. Las entradas pueden ser listas, arreglos o columnas de pandas.

In [ ]:
def proporciones(referencia, actual, k=10):
    """Fracción de la referencia (q) y de la muestra actual (p) en cada cubeta de cuantiles de la referencia."""
    ...


q_temp, p_temp = proporciones(b2011["temp"], b2012["temp"])
print("referencia:", q_temp.round(3))
print("actual:    ", p_temp.round(3))

In [ ]:
grader.check("p1")

## 2. El índice de estabilidad poblacional

Con las fracciones $q_i$ (referencia) y $p_i$ (actual) de cada cubeta, el **PSI** es

$$\mathrm{PSI} = \sum_{i} (p_i - q_i)\,\ln\frac{p_i}{q_i}.$$

Cada sumando es no negativo (si $p_i > q_i$, los dos factores son positivos; si $p_i < q_i$, los dos son negativos), y el PSI vale cero solo si las dos distribuciones coinciden cubeta por cubeta. No es una fórmula inventada: separando la suma,

$$\mathrm{PSI} = \sum_i p_i \ln\frac{p_i}{q_i} + \sum_i q_i \ln\frac{q_i}{p_i} = D_{\mathrm{KL}}(p \,\|\, q) + D_{\mathrm{KL}}(q \,\|\, p),$$

la **divergencia de Kullback–Leibler simetrizada** entre las dos distribuciones discretizadas. En la industria (empezó en el riesgo de crédito) se usan como regla práctica los umbrales 0.1 y 0.25: debajo de 0.1, «estable»; entre 0.1 y 0.25, «cambio moderado, hay que vigilar»; arriba de 0.25, «cambio fuerte». Son convenciones, no teoremas.

Hay un problema: si una cubeta queda **vacía** en una de las dos muestras, $\ln(p_i/q_i)$ es $\pm\infty$. La solución habitual es reemplazar cada fracción por $\max(p_i, \varepsilon)$ con un $\varepsilon$ pequeño antes de calcular.

**Pregunta 2.** Escribe `psi(referencia, actual, k=10, eps=1e-4)`, que devuelve el PSI como `float`. Usa tu función `proporciones` y reemplaza tanto $p_i$ como $q_i$ por $\max(\cdot, \varepsilon)$ (sin volver a normalizar) antes de aplicar la fórmula.

In [ ]:
def psi(referencia, actual, k=10, eps=1e-4):
    """Índice de estabilidad poblacional entre una referencia y una muestra actual."""
    ...


print(f"PSI de la temperatura, todo 2011 contra todo 2012: {psi(b2011['temp'], b2012['temp']):.4f}")

In [ ]:
grader.check("p2")

## 3. El estadístico de Kolmogorov–Smirnov

El PSI depende de cómo se elijan las cubetas. La prueba de **Kolmogorov–Smirnov** de dos muestras no discretiza nada: compara directamente las **funciones de distribución empíricas**,

$$F_a(x) = \frac{\#\{i : a_i \le x\}}{n}, \qquad F_b(x) = \frac{\#\{j : b_j \le x\}}{m},$$

mediante la mayor distancia vertical entre ellas,

$$D = \sup_{x} \bigl|F_a(x) - F_b(x)\bigr| .$$

$D$ está entre 0 y 1 y no depende de la escala del atributo (es invariante ante cualquier transformación creciente). Las dos funciones son escalones que solo cambian en los puntos de las muestras, así que el supremo se alcanza en alguno de los $n + m$ datos: basta evaluar $F_a$ y $F_b$ en la muestra combinada. Con las muestras **ordenadas**, $\#\{i : a_i \le x\}$ es `np.searchsorted(a_ordenada, x, side="right")`, y se puede evaluar en todos los puntos a la vez.

Si las dos muestras vienen de la misma distribución continua, $\sqrt{nm/(n+m)}\,D$ converge a la distribución de Kolmogorov, y de ahí sale el valor $p$. Aquí nos interesa sobre todo $D$ como **tamaño del efecto**: con miles de datos, diferencias minúsculas resultan «significativas».

**Pregunta 3.** Escribe `ks_estadistico(a, b)`, que devuelve el estadístico $D$ de Kolmogorov–Smirnov de dos muestras como `float`, **vectorizado** (sin ciclos de Python y sin `scipy`). Debe funcionar con muestras de tamaños distintos y con valores repetidos (los atributos discretos tienen muchísimos empates). Debe dar lo mismo que `scipy.stats.ks_2samp(a, b).statistic`.

In [ ]:
def ks_estadistico(a, b):
    """Estadístico D de Kolmogorov–Smirnov de dos muestras."""
    ...


print(f"D de la temperatura, 2011 contra 2012: {ks_estadistico(b2011['temp'], b2012['temp']):.4f}")

In [ ]:
grader.check("p3")

## 4. Un reporte de cambio por atributo

En producción no se revisa un atributo a la vez: se arma un **reporte** periódico con todos los atributos (y con las predicciones del modelo) y se ordena para mirar primero lo que más se movió.

**Pregunta 4.** Escribe `reporte_cambio(referencia, actual, columnas, k=10)`. Recibe dos `DataFrame` y una lista de nombres de columnas, y devuelve un `DataFrame` con un renglón por columna (el índice son los nombres) y tres columnas:

- `psi`: el PSI de esa columna (con tu función, $k$ dado y $\varepsilon = 10^{-4}$);
- `ks`: el estadístico $D$ de Kolmogorov–Smirnov;
- `nivel`: el texto `"estable"` si $\mathrm{PSI} < 0.1$, `"moderado"` si $0.1 \le \mathrm{PSI} < 0.25$ y `"fuerte"` si $\mathrm{PSI} \ge 0.25$.

Los renglones van ordenados de **mayor a menor** PSI.

In [ ]:
def reporte_cambio(referencia, actual, columnas, k=10):
    """PSI, estadístico KS y nivel de alerta de cada columna, del mayor PSI al menor."""
    ...


reporte_cambio(b2011, b2012, VARIABLES + ["total"]).round(3)

In [ ]:
grader.check("p4")

### ¿Contra qué se compara?

El reporte de arriba compara **todo 2011** contra **todo 2012**. Ahora hagamos lo que haría el sistema en vivo: al cerrar cada mes de 2012 se arma el reporte de ese mes. Pero hay una trampa. Si comparamos, por ejemplo, la temperatura de **julio de 2012** con la de **todo 2011**, el PSI sale enorme… porque en julio hace calor. Eso no es un cambio del mundo: es la estación del año, y el modelo ya la vio en sus datos de entrenamiento. La referencia correcta para julio de 2012 es **julio de 2011**.

La celda siguiente calcula las dos versiones para la temperatura, y además el PSI de las **predicciones** del modelo y de las **rentas observadas** contra el mismo mes de 2011.

In [ ]:
filas = []
for periodo, actual in b2012.groupby("periodo"):
    mismo_mes = b2011[b2011["mes"] == periodo.month]
    pred_ref, pred_act = modelo.predict(mismo_mes[VARIABLES]), modelo.predict(actual[VARIABLES])
    filas.append({
        "periodo": periodo,
        "temp (contra 2011 completo)": psi(b2011["temp"], actual["temp"]),
        "temp (contra el mismo mes)": psi(mismo_mes["temp"], actual["temp"]),
        "predicciones": psi(pred_ref, pred_act),
        "rentas observadas": psi(mismo_mes["total"], actual["total"]),
        "error absoluto medio": mean_absolute_error(actual["total"], pred_act),
        "error medio (obs. − pred.)": np.mean(actual["total"] - pred_act),
    })
mensual = pd.DataFrame(filas).set_index("periodo")
mensual.round(2)

In [ ]:
meses = np.arange(1, 13)
nombres_mes = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
fig, ax = plt.subplots(figsize=(10, 5))
series = {
    "temperatura (contra 2011 completo)": (mensual["temp (contra 2011 completo)"], rdp.estilo.GRIS),
    "temperatura (contra el mismo mes)": (mensual["temp (contra el mismo mes)"], rdp.estilo.DORADO_OSC),
    "predicciones": (mensual["predicciones"], rdp.estilo.AZUL),
    "rentas observadas": (mensual["rentas observadas"], rdp.estilo.ROJO),
}
for nombre, (valores, color) in series.items():
    ax.plot(meses, valores, color=color, marker="o")
ax.set_yscale("log")
for umbral in [0.1, 0.25]:
    ax.axhline(umbral, color=rdp.estilo.TENUE, lw=1, ls="--")
ax.set_xticks(meses, nombres_mes)
ax.set_xlim(0.6, 16.5)
rdp.estilo.etiquetar(ax, 12.3, {n: v.iloc[-1] for n, (v, _) in series.items()})
ax.set_title("PSI de cada mes de 2012 (escala log; umbrales 0.1 y 0.25)")
ax.set_xlabel("mes de 2012")
ax.set_ylabel("PSI");

Contra el mismo mes de 2011, el PSI de la temperatura baja mucho, aunque no a cero: el clima de un mes no se repite de un año a otro (marzo de 2012 fue excepcionalmente cálido en el este de Estados Unidos). Las predicciones, en cambio, casi no se mueven: el modelo está viendo **entradas parecidas** a las del año anterior y por eso predice más o menos lo mismo. Lo que sí se movió, y mucho, son las **rentas observadas**: en 2012 el sistema creció (más estaciones, más usuarios registrados), así que con el mismo clima y a la misma hora se rentan muchas más bicicletas. Es un **cambio de concepto**: cambió $P(y \mid \mathbf{x})$, no $P(\mathbf{x})$. Ningún monitor de entradas lo puede ver; solo se detecta comparando contra las etiquetas, y el error medio positivo dice que el modelo **subestima** sistemáticamente.

# Parte B · Un servicio de predicción

## 5. Una API con FastAPI

Para que otros sistemas usen el modelo lo envolvemos en un **servicio web**: un programa que recibe una petición HTTP con los atributos de una hora y responde con la predicción. **FastAPI** hace esto con muy poco código, y usa **Pydantic** para declarar el **esquema** de la entrada: qué campos hay, de qué tipo y en qué rango. Si una petición no cumple el esquema, FastAPI la rechaza solo, con el código de estado **422** (*Unprocessable Entity*) y un mensaje que dice qué campo está mal, sin que el modelo llegue a ver el dato malo. Un ejemplo mínimo:

```python
class Punto(BaseModel):
    x: float = Field(ge=0, le=10)    # 0 ≤ x ≤ 10
    n: int = Field(ge=1)             # entero, al menos 1

app = FastAPI()

@app.post("/duplicar")
def duplicar(punto: Punto):
    return {"resultado": 2 * punto.x * punto.n}
```

Dentro de la función, `punto.model_dump()` da un diccionario con los campos. Para probar el servicio **no hace falta levantar un servidor**: `TestClient(app).post("/duplicar", json={"x": 1.5, "n": 2})` simula la petición y devuelve una respuesta con `.status_code` y `.json()`. En producción el mismo archivo se serviría con `uvicorn modulo:app --host 0.0.0.0 --port 8000`.

Una buena práctica: la respuesta incluye siempre la **versión** del modelo que la produjo. Cuando un cliente se queje de una predicción rara, hay que saber qué modelo la hizo.

**Pregunta 5.** Escribe:

- la clase `EntradaHora` (un `BaseModel` de Pydantic) con los seis campos de `VARIABLES` y estos tipos y rangos (inclusivos): `hora` entero de 0 a 23; `laboral` entero 0 o 1; `clima` entero de 1 a 4; `temp` real de −20 a 50; `humedad` real de 0 a 100; `viento` real de 0 a 100. Todos son obligatorios.
- la función `crear_app(modelo, version)`, que devuelve una aplicación `FastAPI` con dos rutas:
  - `POST /predecir`: recibe una `EntradaHora` y responde `{"rentas": r, "version": version}`, donde `r` es la predicción del modelo como `float`. Para predecir arma un `DataFrame` de **un renglón** con las columnas en el orden de `VARIABLES` y llama a `modelo.predict`.
  - `GET /salud`: responde `{"estado": "ok", "version": version}` (lo usan los balanceadores de carga para saber si el servicio está vivo).

In [ ]:
class EntradaHora(BaseModel):
    ...


def crear_app(modelo, version):
    # servicio de predicción de rentas por hora
    app = FastAPI(title="Rentas de bicicletas")
    ...
    return app


cliente = TestClient(crear_app(modelo, "2011.1"))
ejemplo = {"hora": 8, "laboral": 1, "clima": 1, "temp": 22.0, "humedad": 55.0, "viento": 10.0}
respuesta = cliente.post("/predecir", json=ejemplo)
print(respuesta.status_code, respuesta.json())
respuesta = cliente.post("/predecir", json={**ejemplo, "hora": 25})
print(respuesta.status_code, respuesta.json()["detail"][0]["msg"])

In [ ]:
grader.check("p5")

## 6. Reentrenar con una ventana móvil

Ya vimos que el modelo de 2011 subestima las rentas de 2012 y que ningún monitor de entradas lo avisa. Una política simple y muy común es **reentrenar periódicamente**: al empezar cada mes, se entrena un modelo nuevo con los últimos $w$ meses de datos etiquetados y ese modelo atiende el mes. La ventana móvil olvida el pasado lejano a propósito: si el mundo cambió, los datos viejos estorban.

Ojo con la **fuga de información**: para atender el mes $t$ solo se pueden usar los meses **anteriores** a $t$. Aquí suponemos que las etiquetas llegan de inmediato (al terminar el mes ya sabemos cuántas bicicletas se rentaron); en la reflexión 1 veremos qué pasa cuando no.

**Pregunta 6.** Escribe `reentrenar_ventana(datos, fabrica, meses_ventana=12, primer_mes="2012-01")`. `datos` es un `DataFrame` como `bicis` (con las columnas `periodo`, las de `VARIABLES` y `total`) y `fabrica` es una función sin argumentos que devuelve un modelo nuevo sin entrenar (como `fabrica_modelo`). Para cada mes $t$ presente en `datos` a partir de `primer_mes` (inclusive), en orden:

1. entrena `fabrica()` con los renglones cuyo periodo está entre $t - w$ y $t - 1$ (los $w$ = `meses_ventana` meses **anteriores** a $t$), usando `VARIABLES` como entrada y `total` como respuesta;
2. predice los renglones del mes $t$ y calcula el error absoluto medio.

Devuelve una `pd.Series` con el error de cada mes, con los periodos como índice. Para restar meses a un periodo de pandas basta `t - 3`, y `pd.Period("2012-01", freq="M")` convierte el texto en periodo.

In [ ]:
def reentrenar_ventana(datos, fabrica, meses_ventana=12, primer_mes="2012-01"):
    """Error absoluto medio de cada mes con un modelo reentrenado en los meses_ventana meses anteriores."""
    ...


inicio = time.perf_counter()
error_movil = reentrenar_ventana(bicis, fabrica_modelo, meses_ventana=12)
print(f"tiempo: {time.perf_counter() - inicio:.1f} s")
pd.DataFrame({"modelo fijo de 2011": mensual["error absoluto medio"], "ventana de 12 meses": error_movil}).round(1)

In [ ]:
grader.check("p6")

Comparemos mes a mes el error del modelo que nunca se tocó con el del que se reentrena cada mes.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(meses, mensual["error absoluto medio"], color=rdp.estilo.ROJO, marker="o")
ax.plot(meses, error_movil.to_numpy(), color=rdp.estilo.AZUL, marker="o")
ax.set_xticks(meses, nombres_mes)
ax.set_xlim(0.6, 15)
ax.set_ylim(0, None)
rdp.estilo.etiquetar(ax, 12.3, {"modelo fijo de 2011": mensual["error absoluto medio"].iloc[-1],
                                "reentrenado (12 meses)": error_movil.iloc[-1]})
ax.set_title("Error absoluto medio de cada mes de 2012")
ax.set_xlabel("mes de 2012")
ax.set_ylabel("rentas por hora");
print(f"error medio anual · fijo: {mensual['error absoluto medio'].mean():.1f} · reentrenado: {error_movil.mean():.1f}")

En enero las dos curvas coinciden: el modelo reentrenado para enero de 2012 usa exactamente los doce meses de 2011, igual que el fijo. A partir de ahí el reentrenado va incorporando el crecimiento del sistema y su error se separa del fijo, sobre todo en el otoño. Pero no llega al error de entrenamiento: una ventana de doce meses mezcla el sistema de hace un año con el de hoy, y el modelo sigue llegando tarde a un mundo que crece. ¿Una ventana más corta lo arreglaría, o lo haría más ruidoso? Puedes probarlo con tu función.

## 7. Reflexión

Estas preguntas no se verifican automáticamente: respóndelas en la celda de texto que sigue a cada una. Si necesitas hacer cuentas, agrega celdas de código.

<!-- BEGIN QUESTION -->

**Reflexión 1.** En la pregunta 6 supusimos que las rentas reales de un mes se conocen en cuanto el mes termina. Imagina ahora un modelo de riesgo de crédito, como el de los clientes de Taiwán del laboratorio 3: un cliente «incumple» si no paga en los meses siguientes, así que la etiqueta llega con **tres meses de retraso**. (a) Modifica mentalmente (o en código, con un argumento `retraso`) tu `reentrenar_ventana`: ¿qué meses puede usar el modelo que atiende el mes $t$? Si quieres, mide en las bicicletas qué le pasa al error con un retraso de tres meses. (b) Mientras llegan las etiquetas, ¿qué puedes vigilar y qué **no** puedes detectar con eso? Relaciónalo con lo que viste en la gráfica de PSI de esta libreta. (c) Propón una política concreta de monitoreo y reentrenamiento para ese caso.

_Escribe aquí tu respuesta._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Reflexión 2.** Calcula el valor $p$ de la prueba de Kolmogorov–Smirnov (`stats.ks_2samp`) para la **humedad** de cada mes de 2012 contra el mismo mes de 2011, y compáralo con el PSI y con $D$. ¿Cuántos meses serían «significativos» al 5 %? Ahora toma dos submuestras **del mismo mes de 2011** (por ejemplo, las horas pares contra las impares, o la primera quincena contra la segunda) y repite: ¿qué tan fácil es obtener un valor $p$ pequeño sin que haya ningún cambio de fondo? ¿Por qué las horas de un mismo mes no son observaciones independientes, y qué le hace eso a la prueba? Con base en esto, ¿usarías el valor $p$ o el tamaño del efecto ($D$ o PSI) para disparar una alerta, y por qué la elección de la **referencia** (todo el año contra el mismo mes) importa tanto como la del umbral?

_Escribe aquí tu respuesta._

<!-- END QUESTION -->



---

To double-check your work, the cell below will rerun all of the autograder tests.

In [ ]:
grader.check_all()